# Speed Limit Signs -- YOLOv5n Training (Kaggle)

Train YOLOv5n for the AI-Powered HUD project.

**Data source:** [MTSD](https://www.mapillary.com/dataset/trafficsign) (Mapillary Traffic Sign Dataset)

**Target device:** Luckfox Pico Ultra (RV1106G3, 0.5 TOPS NPU, INT8)

**Model:** 11-class universal, covers AU + CN speed limits

**Training resolution:** 640x640

**Runtime:** Enable GPU (Settings > Accelerator > GPU T4 x2)

---

### Kaggle vs Colab differences

| | Colab | Kaggle |
|---|---|---|
| Dataset | Google Drive / browser upload | Kaggle Datasets (Add Data) |
| Checkpoint backup | Google Drive sync thread | `/kaggle/working/` (auto-saved as Output) |
| Download results | `files.download()` | Output tab after notebook completes |
| GPU quota | Usage limits (variable) | 30h/week (T4 x2 or P100) |
| Session limit | ~12h (disconnects often) | 12h (more stable) |

### Resume after session timeout

1. After training completes (or times out), download `last.pt` from the Output tab
2. Upload it as a new Kaggle Dataset (e.g. `ai-hud-checkpoint`)
3. Add that dataset via **Add Data** in the right sidebar
4. Re-run all cells -- checkpoint will be auto-detected and training resumes

---

| ID | Class | AU | CN |
|----|-------|:--:|:--:|
| 0 | speed_sign_20 | - | Y |
| 1 | speed_sign_30 | Y | Y |
| 2 | speed_sign_40 | Y | Y |
| 3 | speed_sign_50 | Y | Y |
| 4 | speed_sign_60 | Y | Y |
| 5 | speed_sign_70 | Y | Y |
| 6 | speed_sign_80 | Y | Y |
| 7 | speed_sign_90 | Y | - |
| 8 | speed_sign_100 | Y | Y |
| 9 | speed_sign_110 | Y | Y |
| 10 | speed_sign_120 | - | Y |

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
TARGET_CLASSES = [
    "speed_sign_20",  "speed_sign_30",  "speed_sign_40",
    "speed_sign_50",  "speed_sign_60",  "speed_sign_70",
    "speed_sign_80",  "speed_sign_90",  "speed_sign_100",
    "speed_sign_110", "speed_sign_120",
]
DATASET_DIR = "speed_signs_dataset"
PROJECT_NAME = "speed_signs"
RUN_NAME = "universal"  # YOLOv5 run directory name

NC = len(TARGET_CLASSES)
WORK_DIR = "/kaggle/working"
DATASET_ROOT = f"{WORK_DIR}/{DATASET_DIR}"

# ============================================================
# TRAINING PARAMETERS
# ============================================================
EPOCHS = 300
BATCH_SIZE = 64   # Single T4 (15GB), YOLOv5n uses ~5-6GB @ bs64
IMG_SIZE = 640
WORKERS = 8       # Maximize data loading throughput to keep GPU fed

# Use single GPU: YOLOv5n is too small (1.7M params) to benefit from
# multi-GPU DP mode. DP communication overhead actually slows it down.
DEVICE = "0"

print(f"Model:     {PROJECT_NAME} (11-class universal, AU + CN)")
print(f"Classes:   {NC}")
print(f"Training:  {EPOCHS} epochs, batch {BATCH_SIZE}, img {IMG_SIZE}")
print(f"Device:    GPU {DEVICE} (single T4, DP overhead > compute for YOLOv5n)")
print(f"Dataset:   {DATASET_ROOT}")
print(f"Output:    {WORK_DIR} (download from Output tab)")
print()
for i, name in enumerate(TARGET_CLASSES):
    print(f"  {i}: {name}")

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
        print(f"  VRAM: {vram / 1024**3:.1f} GB")

In [ ]:
import os

# Clone airockchip/yolov5 (RKNN-optimized fork, required for --rknpu export)
YOLOV5_DIR = f"{WORK_DIR}/yolov5"
if os.path.exists(f"{YOLOV5_DIR}/.git"):
    print("yolov5 repo already exists, skipping clone.")
else:
    !git clone https://github.com/airockchip/yolov5.git {YOLOV5_DIR}

%cd {YOLOV5_DIR}
!pip install -r requirements.txt -q

# [Fix] onnxscript is required by ONNX export with PyTorch >= 2.6
!pip install onnxscript -q

# [Fix] Pillow 10+ removed font.getsize() used by YOLOv5 utils/plots.py
!pip install "Pillow<10" -q

# [Fix] albumentations 2.0+ has breaking API changes (RandomSizedBBoxSafeCrop requires `size`)
!pip install "albumentations<2.0" -q

## 2. Load Dataset

### How to upload your dataset to Kaggle:

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) > **New Dataset**
2. Upload `speed_signs_augmented.tar.gz` (or the uncompressed directory)
3. Name it e.g. `ai-hud-speed-signs`
4. In this notebook, click **Add Data** (right sidebar) and search for your dataset
5. It will appear at `/kaggle/input/ai-hud-speed-signs/`

The cell below auto-detects the dataset from `/kaggle/input/`.

In [ ]:
import os, yaml, shutil
from pathlib import Path
from collections import defaultdict

# ============================================================
# Auto-detect dataset from /kaggle/input/
# Handles both:
#   a) Uploaded as tar.gz -> extract first
#   b) Uploaded as directory -> use directly
# ============================================================
INPUT_DIR = "/kaggle/input"

def find_dataset():
    """Search /kaggle/input/ for the dataset (tar.gz or directory with data.yaml)."""
    # Search for tar.gz files first
    for root, dirs, files in os.walk(INPUT_DIR):
        for f in files:
            if f.endswith('.tar.gz') and 'speed_sign' in f.lower():
                return 'tar', os.path.join(root, f)

    # Search for data.yaml (directory upload)
    for root, dirs, files in os.walk(INPUT_DIR):
        if 'data.yaml' in files:
            # Verify it has train/val subdirs
            if (os.path.isdir(os.path.join(root, 'train')) or
                os.path.isdir(os.path.join(root, 'val'))):
                return 'dir', root

    return None, None

dtype, dpath = find_dataset()

if dtype == 'tar':
    print(f"Found tar.gz: {dpath}")
    print("Extracting...")
    !tar -xzf "{dpath}" -C {WORK_DIR}/
    # Auto-detect extracted directory
    if not os.path.exists(DATASET_ROOT):
        extracted = [
            d for d in os.listdir(WORK_DIR)
            if os.path.isdir(f"{WORK_DIR}/{d}")
            and os.path.exists(f"{WORK_DIR}/{d}/data.yaml")
            and d != DATASET_DIR
            and d != 'yolov5'
        ]
        if len(extracted) == 1:
            src = f"{WORK_DIR}/{extracted[0]}"
            print(f"Renaming: {extracted[0]} -> {DATASET_DIR}")
            os.rename(src, DATASET_ROOT)
        elif len(extracted) > 1:
            raise RuntimeError(f"Multiple dataset directories found: {extracted}")
        else:
            raise FileNotFoundError("No directory with data.yaml found after extraction.")

elif dtype == 'dir':
    # Kaggle input is read-only, copy to working dir for mutability
    print(f"Found dataset directory: {dpath}")
    if not os.path.exists(DATASET_ROOT):
        print(f"Copying to {DATASET_ROOT} (Kaggle input is read-only)...")
        shutil.copytree(dpath, DATASET_ROOT)
    else:
        print(f"Dataset already at {DATASET_ROOT}")

else:
    raise FileNotFoundError(
        "Dataset not found in /kaggle/input/.\n"
        "Please add your dataset via the 'Add Data' button in the right sidebar.\n"
        "Upload speed_signs_augmented.tar.gz as a Kaggle Dataset first.")

# Verify
for split in ["train", "val"]:
    img_dir = f"{DATASET_ROOT}/{split}/images"
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"  {split}: {n} images")

# Update paths in data.yaml for Kaggle absolute paths
data_yaml = {
    "train": f"{DATASET_ROOT}/train/images",
    "val": f"{DATASET_ROOT}/val/images",
    "nc": NC,
    "names": TARGET_CLASSES,
}
with open(f"{DATASET_ROOT}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"\nDataset ready at {DATASET_ROOT}")

In [ ]:
# ============================================================
# Pre-resize images to max 1280px long edge for faster training I/O.
# YOLO labels are normalized (0-1), so no label changes needed.
#
# Without this, augmented crops can be 2000-4000px from original MTSD
# images, causing slow JPEG decode during training (~1.5s/batch).
# After resize: JPEG decode is ~3-5x faster (~0.3-0.5s/batch).
# ============================================================
import cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

MAX_LONG_EDGE = 1280

def resize_if_needed(img_path):
    """Resize image in-place if long edge > MAX_LONG_EDGE."""
    img = cv2.imread(str(img_path))
    if img is None:
        return False
    h, w = img.shape[:2]
    long_edge = max(h, w)
    if long_edge <= MAX_LONG_EDGE:
        return False
    scale = MAX_LONG_EDGE / long_edge
    new_w, new_h = int(w * scale), int(h * scale)
    img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(img_path), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return True

for split in ["train", "val"]:
    img_dir = Path(f"{DATASET_ROOT}/{split}/images")
    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    with ThreadPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(resize_if_needed, images))
    resized = sum(results)
    print(f"  {split}: {resized}/{len(images)} images resized (max {MAX_LONG_EDGE}px)")

## 3. Dataset Statistics & Visualization

In [ ]:
# Class distribution
total_stats = defaultdict(int)

for split in ["train", "val"]:
    lbl_dir = f"{DATASET_ROOT}/{split}/labels"
    img_dir = f"{DATASET_ROOT}/{split}/images"
    if not os.path.exists(lbl_dir):
        continue

    n_images = len([f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    split_counts = defaultdict(int)

    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith(".txt"):
            continue
        try:
            with open(os.path.join(lbl_dir, lbl_file), "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        split_counts[cls_id] += 1
                        total_stats[cls_id] += 1
        except (UnicodeDecodeError, ValueError):
            continue

    print(f"\n  {split}: {n_images} images")
    for cls_id in range(NC):
        print(f"    {cls_id}: {TARGET_CLASSES[cls_id]:<18s} {split_counts.get(cls_id, 0):>5d}")

print(f"\n{'='*55}")
total_ann = sum(total_stats.values())
for cls_id in range(NC):
    name = TARGET_CLASSES[cls_id]
    count = total_stats.get(cls_id, 0)
    pct = (count / total_ann * 100) if total_ann > 0 else 0
    bar = '#' * min(count // 10, 40)
    print(f"  {cls_id}: {name:<18s} {count:>5d} ({pct:>5.1f}%) {bar}")
print(f"  {'TOTAL':<21s} {total_ann:>5d}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

COLORS = [
    (255, 80, 80), (80, 255, 80), (80, 80, 255), (255, 255, 80),
    (255, 80, 255), (80, 255, 255), (255, 160, 80), (160, 80, 255),
    (80, 160, 255), (200, 200, 80), (200, 80, 200),
]

def draw_yolo_boxes(img_path, label_path, class_names):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if cls_id < len(class_names) else f"cls{cls_id}"
                cv2.putText(img, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

train_img_dir = f"{DATASET_ROOT}/train/images"
train_lbl_dir = f"{DATASET_ROOT}/train/labels"
all_imgs = [f for f in os.listdir(train_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if all_imgs:
    samples = random.sample(all_imgs, min(8, len(all_imgs)))
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_file in zip(axes.flat, samples):
        stem = Path(img_file).stem
        img = draw_yolo_boxes(
            os.path.join(train_img_dir, img_file),
            os.path.join(train_lbl_dir, stem + ".txt"),
            TARGET_CLASSES,
        )
        if img is not None:
            ax.imshow(img)
            ax.set_title(img_file[:30], fontsize=8)
        ax.axis("off")
    plt.suptitle("Training Samples", fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Train YOLOv5n

Transfer learning from COCO-pretrained weights.

Expected training time: ~2-3 hours on T4 GPU for 300 epochs.

**Checkpoint resume:**

Kaggle saves `/kaggle/working/` as notebook Output. To resume after a session timeout:

1. Download `last.pt` from the Output tab of your previous run
2. Create a new Kaggle Dataset with `last.pt` (and optionally `best.pt`)
3. Add that dataset via **Add Data** in this notebook
4. Re-run all cells -- the checkpoint is auto-detected from `/kaggle/input/`

**Resume priority:** local `last.pt` > checkpoint from Kaggle input > fresh start.

**Compatibility fixes applied automatically:**
- PyTorch >= 2.6: `torch.load()` patched to `weights_only=False`
- Pillow 10+: `font.getsize()` patched to `font.getbbox()`
- PyTorch >= 2.4: `torch.cuda.amp.autocast()` -> `torch.amp.autocast()`
- PyTorch >= 2.4: `torch.cuda.amp.GradScaler()` -> `torch.amp.GradScaler()`
- PyTorch >= 2.4: `amp.autocast()` alias pattern patched
- albumentations >= 2.0: pinned to `<2.0` to avoid breaking API changes

In [ ]:
%cd {YOLOV5_DIR}

import os, re, shutil, subprocess

# ============================================================
# Paths
# ============================================================
RUN_DIR = f"{YOLOV5_DIR}/runs/{PROJECT_NAME}/{RUN_NAME}"
WEIGHTS_DIR = f"{RUN_DIR}/weights"
LOCAL_LAST = f"{WEIGHTS_DIR}/last.pt"
LOCAL_BEST = f"{WEIGHTS_DIR}/best.pt"

# Output copies (easy to find in Output tab)
OUTPUT_DIR = f"{WORK_DIR}/checkpoints"

# ============================================================
# [1/3] Checkpoint resume detection
# Priority: local last.pt > Kaggle input checkpoint > fresh start
# ============================================================
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

resume_from = None

if os.path.exists(LOCAL_LAST):
    size_mb = os.path.getsize(LOCAL_LAST) / 1024 / 1024
    print(f"[Resume] Found local checkpoint: {LOCAL_LAST} ({size_mb:.1f} MB)")
    resume_from = LOCAL_LAST
else:
    # Search /kaggle/input/ for a checkpoint dataset
    ckpt_found = None
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f == "last.pt":
                ckpt_found = os.path.join(root, f)
                break
        if ckpt_found:
            break

    if ckpt_found:
        size_mb = os.path.getsize(ckpt_found) / 1024 / 1024
        print(f"[Resume] Found checkpoint in Kaggle input: {ckpt_found} ({size_mb:.1f} MB)")
        print(f"[Resume] Copying to {LOCAL_LAST}...")
        shutil.copy2(ckpt_found, LOCAL_LAST)
        # Also copy best.pt if available next to last.pt
        best_candidate = os.path.join(os.path.dirname(ckpt_found), "best.pt")
        if os.path.exists(best_candidate):
            shutil.copy2(best_candidate, LOCAL_BEST)
        resume_from = LOCAL_LAST
        print(f"[Resume] Checkpoint restored.")
    else:
        print("[Resume] No checkpoint found. Starting fresh training.")

# ============================================================
# [2/3] Compatibility patches (idempotent)
# Mirrors training/patch_yolov5_compat.py (inline for portability)
# ============================================================

# --- Patch 1: torch.load weights_only ---
def _patch_torch_load(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    original = content
    result = []
    i = 0
    while i < len(content):
        idx = content.find('torch.load(', i)
        if idx == -1:
            result.append(content[i:])
            break
        result.append(content[i:idx])
        start = idx + len('torch.load(')
        depth = 1
        j = start
        while j < len(content) and depth > 0:
            if content[j] == '(': depth += 1
            elif content[j] == ')': depth -= 1
            j += 1
        args_str = content[start:j-1]
        if 'weights_only' not in args_str:
            result.append(f'torch.load({args_str}, weights_only=False)')
        else:
            result.append(f'torch.load({args_str})')
        i = j
    content = ''.join(result)
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        return True
    return False

files_out = subprocess.run(
    ["grep", "-rl", "torch.load", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")
patched = 0
for f in files_out:
    if f.endswith('.py'):
        if _patch_torch_load(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"[Patch] torch.load: {patched} files fixed")

# --- Patch 2: Pillow getsize -> getbbox ---
plots_py = "utils/plots.py"
with open(plots_py, 'r') as f:
    full_text = f.read()

if 'getbbox' in full_text:
    print("[Patch] Pillow getsize: already applied")
elif 'getsize' not in full_text:
    print("[Patch] Pillow getsize: not needed")
else:
    lines = full_text.splitlines(keepends=True)
    new_lines = []
    for line in lines:
        if 'self.font.getsize(label)' in line and 'try' not in line:
            indent = line[:len(line) - len(line.lstrip())]
            inner = indent + '    '
            new_lines.append(f'{indent}try:\n')
            new_lines.append(f'{inner}w, h = self.font.getsize(label)\n')
            new_lines.append(f'{indent}except AttributeError:\n')
            new_lines.append(f'{inner}bbox = self.font.getbbox(label)\n')
            new_lines.append(f'{inner}w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]\n')
        else:
            new_lines.append(line)
    with open(plots_py, 'w') as f:
        f.writelines(new_lines)
    print("[Patch] Pillow getsize: applied")

# --- Patch 3: torch.cuda.amp.autocast deprecation ---
def _patch_autocast(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    if 'torch.cuda.amp.autocast' not in content:
        return False
    def _replace(match):
        args = match.group(1).strip()
        if not args:
            return 'torch.amp.autocast("cuda")'
        if '=' in args:
            return f'torch.amp.autocast("cuda", {args})'
        return f'torch.amp.autocast("cuda", enabled={args})'
    new_content = re.sub(
        r'torch\.cuda\.amp\.autocast\(([^)]*)\)',
        _replace, content,
    )
    if new_content != content:
        with open(filepath, 'w') as f:
            f.write(new_content)
        return True
    return False

files_out = subprocess.run(
    ["grep", "-rl", "torch.cuda.amp.autocast", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")
patched = 0
for f in files_out:
    if f and f.endswith('.py'):
        if _patch_autocast(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"[Patch] autocast: {patched} files fixed")

# --- Patch 4: torch.cuda.amp.GradScaler deprecation ---
def _patch_gradscaler(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    if 'torch.cuda.amp.GradScaler' not in content:
        return False
    def _replace(match):
        args = match.group(1).strip()
        if not args:
            return 'torch.amp.GradScaler("cuda")'
        if '=' in args:
            return f'torch.amp.GradScaler("cuda", {args})'
        return f'torch.amp.GradScaler("cuda", enabled={args})'
    new_content = re.sub(
        r'torch\.cuda\.amp\.GradScaler\(([^)]*)\)',
        _replace, content,
    )
    if new_content != content:
        with open(filepath, 'w') as f:
            f.write(new_content)
        return True
    return False

files_out = subprocess.run(
    ["grep", "-rl", "torch.cuda.amp.GradScaler", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")
patched = 0
for f in files_out:
    if f and f.endswith('.py'):
        if _patch_gradscaler(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"[Patch] GradScaler: {patched} files fixed")

# --- Patch 5: amp alias pattern (from torch.cuda import amp) ---
def _patch_amp_alias(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    if not re.search(r'from\s+torch\.cuda\s+import\s+amp', content):
        return False
    original = content
    def _replace_ac(match):
        args = match.group(1).strip()
        if not args:
            return 'torch.amp.autocast("cuda")'
        if '=' in args:
            return f'torch.amp.autocast("cuda", {args})'
        return f'torch.amp.autocast("cuda", enabled={args})'
    content = re.sub(
        r'(?<!\w)(?<!\.)amp\.autocast\(([^)]*)\)',
        _replace_ac, content,
    )
    def _replace_gs(match):
        args = match.group(1).strip()
        if not args:
            return 'torch.amp.GradScaler("cuda")'
        if '=' in args:
            return f'torch.amp.GradScaler("cuda", {args})'
        return f'torch.amp.GradScaler("cuda", enabled={args})'
    content = re.sub(
        r'(?<!\w)(?<!\.)amp\.GradScaler\(([^)]*)\)',
        _replace_gs, content,
    )
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        return True
    return False

files_out = subprocess.run(
    ["grep", "-rl", "amp.autocast", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")
patched = 0
for f in files_out:
    if f and f.endswith('.py'):
        if _patch_amp_alias(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"[Patch] amp alias: {patched} files fixed")

# ============================================================
# [3/3] Train (auto-resume or fresh start)
# --device 0: single GPU to avoid DP overhead (YOLOv5n too small for multi-GPU)
# No --cache flag: images read from JPEG on-the-fly.
# --cache ram OOMs on Kaggle (13GB shared), --cache disk needs 65GB+.
# ============================================================
print()
if resume_from:
    print(f">>> Resuming training from checkpoint")
    !python train.py --resume "{resume_from}" --device {DEVICE}
else:
    print(f">>> Starting fresh training: {EPOCHS} epochs on GPU {DEVICE}")
    !python train.py \
        --data "{DATASET_ROOT}/data.yaml" \
        --cfg yolov5n.yaml \
        --weights yolov5n.pt \
        --img {IMG_SIZE} \
        --batch-size {BATCH_SIZE} \
        --epochs {EPOCHS} \
        --workers {WORKERS} \
        --device {DEVICE} \
        --project runs/{PROJECT_NAME} \
        --name {RUN_NAME} \
        --exist-ok

# ============================================================
# Copy checkpoints to output directory for easy download
# ============================================================
backed_up = []
for src, name in [(LOCAL_LAST, "last.pt"), (LOCAL_BEST, "best.pt")]:
    if os.path.exists(src):
        dst = f"{OUTPUT_DIR}/{name}"
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        backed_up.append(f"  {name} ({size_mb:.1f} MB)")

results_csv = f"{RUN_DIR}/results.csv"
if os.path.exists(results_csv):
    shutil.copy2(results_csv, f"{OUTPUT_DIR}/results.csv")
    backed_up.append("  results.csv")

if backed_up:
    print(f"\n[Output] Checkpoints saved to {OUTPUT_DIR}:")
    for item in backed_up:
        print(item)
    print(f"\nDownload from the Output tab after notebook completes.")
    print(f"To resume: upload last.pt as a Kaggle Dataset and add it via Add Data.")

## 5. Evaluate Results

In [ ]:
from IPython.display import Image, display

results_dir = f"{YOLOV5_DIR}/runs/{PROJECT_NAME}/{RUN_NAME}"

for img_name, title in [
    ("results.png", "Training Curves"),
    ("confusion_matrix.png", "Confusion Matrix"),
    ("PR_curve.png", "PR Curve"),
    ("F1_curve.png", "F1 Curve"),
]:
    img_path = f"{results_dir}/{img_name}"
    if os.path.exists(img_path):
        print(f"\n{title}:")
        display(Image(filename=img_path, width=700))

In [ ]:
# Validation
!python val.py \
    --data "{DATASET_ROOT}/data.yaml" \
    --weights runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt \
    --img 640 \
    --task val \
    --verbose

In [ ]:
# Visual detection results on val set
!python detect.py \
    --weights runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source "{DATASET_ROOT}/val/images" \
    --project runs/{PROJECT_NAME} \
    --name val_detect \
    --exist-ok \
    --save-txt \
    --max-det 20

detect_dir = f"{YOLOV5_DIR}/runs/{PROJECT_NAME}/val_detect"
if os.path.exists(detect_dir):
    det_imgs = [f for f in os.listdir(detect_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:8]
    if det_imgs:
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        for ax, img_file in zip(axes.flat, det_imgs):
            img = cv2.imread(os.path.join(detect_dir, img_file))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        plt.suptitle("Detection Results (Val Set)", fontsize=14)
        plt.tight_layout()
        plt.show()

## 6. Export ONNX for RKNN

Export with `--rknpu` flag (required for airockchip fork).

In [ ]:
%cd {YOLOV5_DIR}

WEIGHTS = f"runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"

!python export.py \
    --weights {WEIGHTS} \
    --img-size 640 640 \
    --batch-size 1 \
    --rknpu \
    --include onnx

onnx_path = WEIGHTS.replace(".pt", ".onnx")
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"\nONNX exported: {onnx_path} ({size_mb:.2f} MB)")
else:
    print("ONNX export failed! If 'No module named onnxscript', run: !pip install onnxscript")

## 7. Convert to RKNN (INT8 Quantization)

Uses isolated virtualenv to avoid dependency conflicts with Kaggle's pre-installed packages.

In [ ]:
%%writefile {WORK_DIR}/convert_rknn.py
import glob, os
from rknn.api import RKNN

# These are set via environment variables from the calling cell
ONNX_PATH = os.environ["ONNX_PATH"]
RKNN_PATH = os.environ["RKNN_PATH"]
DATASET_ROOT = os.environ["DATASET_ROOT"]

cal_images = sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.jpg"))[:50]
cal_file = f"{os.path.dirname(RKNN_PATH)}/dataset.txt"
with open(cal_file, "w") as f:
    f.write("\n".join(cal_images))
print(f"Calibration images: {len(cal_images)}")

rknn = RKNN(verbose=False)
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform="rv1106",
)

print("Loading ONNX...")
ret = rknn.load_onnx(model=ONNX_PATH)
assert ret == 0, f"Load ONNX failed: {ret}"

print("Building RKNN (INT8 quantization)...")
ret = rknn.build(do_quantization=True, dataset=cal_file)
assert ret == 0, f"Build failed: {ret}"

print("Exporting...")
ret = rknn.export_rknn(RKNN_PATH)
assert ret == 0, f"Export failed: {ret}"

rknn.release()
size_mb = os.path.getsize(RKNN_PATH) / 1024 / 1024
print(f"\nDone! {RKNN_PATH}: {size_mb:.2f} MB")

In [ ]:
import os

RKNN_NAME = "speed_signs_rv1106.rknn"
os.environ["ONNX_PATH"] = f"{YOLOV5_DIR}/runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.onnx"
os.environ["RKNN_PATH"] = f"{WORK_DIR}/{RKNN_NAME}"
os.environ["DATASET_ROOT"] = DATASET_ROOT

!pip install virtualenv -q
!virtualenv {WORK_DIR}/rknn_venv 2>/dev/null || (rm -rf {WORK_DIR}/rknn_venv && virtualenv {WORK_DIR}/rknn_venv)
!{WORK_DIR}/rknn_venv/bin/pip install "setuptools<70" -q
!{WORK_DIR}/rknn_venv/bin/pip install "onnx==1.16.2" -q
!{WORK_DIR}/rknn_venv/bin/pip install rknn-toolkit2 -q

!{WORK_DIR}/rknn_venv/bin/python3 {WORK_DIR}/convert_rknn.py

## 8. Collect Output Artifacts

All artifacts are collected to `/kaggle/working/` for download via the **Output** tab.

After notebook completes:
1. Go to the **Output** tab at the top of the notebook
2. Download the files you need
3. Deploy: `adb push speed_signs_rv1106.rknn /root/model/`

In [ ]:
import shutil, os

# Copy best weights with clear names to output root
weights_dir = f"{YOLOV5_DIR}/runs/{PROJECT_NAME}/{RUN_NAME}/weights"
pt_src = f"{weights_dir}/best.pt"
onnx_src = f"{weights_dir}/best.onnx"

artifacts = {
    f"{WORK_DIR}/speed_signs.pt": (pt_src, "PyTorch weights"),
    f"{WORK_DIR}/speed_signs.onnx": (onnx_src, "ONNX model"),
    f"{WORK_DIR}/{RKNN_NAME}": (f"{WORK_DIR}/{RKNN_NAME}", "RKNN model (INT8, RV1106)"),
}

print(f"Output artifacts (download from Output tab):")
print("=" * 55)

for dst, (src, desc) in artifacts.items():
    if os.path.exists(src):
        if src != dst:
            shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"  {os.path.basename(dst)}: {size_mb:.2f} MB ({desc})")
    else:
        print(f"  {os.path.basename(dst)}: NOT FOUND ({desc})")

# Also save checkpoints for resume
print(f"\nCheckpoints (for resume):")
print("=" * 55)
for name in ["last.pt", "best.pt"]:
    src = f"{weights_dir}/{name}"
    dst = f"{OUTPUT_DIR}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"  checkpoints/{name}: {size_mb:.1f} MB")

print(f"\n{'='*55}")
print(f"  Deployment:")
print(f"{'='*55}")
print(f"  1. adb push {RKNN_NAME} /root/model/")
print(f"  2. adb push build/ai-hud /root/ai-hud")
print(f"\n  Resume training:")
print(f"  1. Download checkpoints/last.pt from Output tab")
print(f"  2. Upload as Kaggle Dataset (e.g. 'ai-hud-checkpoint')")
print(f"  3. Add Data -> select that dataset -> re-run this notebook")

In [ ]:
# ============================================================
# Cleanup: remove large directories to minimize Output size.
# Only keep model artifacts and checkpoints (~20MB total).
# Without cleanup, Output would include yolov5 repo + dataset (~3GB+).
# ============================================================
import shutil, os

keep_files = {
    "speed_signs.pt", "speed_signs.onnx", "speed_signs_rv1106.rknn",
    "checkpoints",  # directory with last.pt + best.pt + results.csv
    "convert_rknn.py",
}

removed = []
for item in os.listdir(WORK_DIR):
    if item in keep_files:
        continue
    path = os.path.join(WORK_DIR, item)
    try:
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
        removed.append(item)
    except Exception as e:
        print(f"  Warning: could not remove {item}: {e}")

print(f"[Cleanup] Removed {len(removed)} items from /kaggle/working/")
for r in removed:
    print(f"  - {r}")

# Show final output contents
print(f"\nFinal Output contents:")
total_size = 0
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp)
        total_size += sz
        rel = os.path.relpath(fp, WORK_DIR)
        print(f"  {rel}: {sz / 1024 / 1024:.2f} MB")
print(f"\n  Total: {total_size / 1024 / 1024:.1f} MB")